# 01: All 18 MedMNIST datasets, 5 runs each (20 epochs)

Reproduces the ResNet-50 (224) results for the 12 MedMNIST2D datasets (Table 3 of the
MedMNIST v2 paper) and the ResNet-50+3D results for the 6 MedMNIST3D datasets (Table 4),
with 5 independent runs per dataset, using the authors' released training code
([MedMNIST/experiments](https://github.com/MedMNIST/experiments)).

## Settings

- **2D:** `--model_flag resnet50 --size 28 --resize --as_rgb`, batch size 128.
- **3D:** `--model_flag resnet50 --conv Conv3d --pretrained_3d i3d --as_rgb`, batch size 32,
  plus `--shape_transform` for AdrenalMNIST3D and VesselMNIST3D only, as in the paper.
- **20 epochs instead of the paper's 100**, because of the compute time available. The
  released script puts its two learning-rate drops at 50% and 75% of the epoch count, so this
  is the same schedule, compressed. Notebook 02 re-runs the datasets that missed at 100 epochs.
- Test AUC comes from the checkpoint with the best validation AUC. No fixed seeds, so every
  run is independent.

## Changes to the released code

- **GPU selection (Cell 4).** The script masks the chosen GPU with `CUDA_VISIBLE_DEVICES` and
  then asks for `cuda:<original index>`, which only works for GPU 0. The fix asks for `cuda:0`
  after masking. It changes which GPU a run uses, nothing about the model, data or training.
- **Output folders.** 3D runs pass `--output_root output/<run>` (an existing flag), so two runs
  that start in the same second don't write into the same timestamped folder.

## Cells

1. GPU check
2. Install packages
3. Download the authors' code
4. GPU-selection fix
5. Configuration and command builder
6. Timing calibration (5 epochs of ChestMNIST: 5.9 min per epoch on one RTX 3090)
7. Main loop: two runs at a time, one per GPU, skipping runs that already finished
8. Summary table, also saved as `results_summary.csv`

The saved output of Cell 7 is from the last pass. Earlier passes had already finished the 2D
datasets, so those show as skipped there. Cell 8 summarises all 90 runs.

In [1]:
# Cell 1: confirm both GPUs are visible before doing anything else.
import torch

print("CUDA available:", torch.cuda.is_available())
n_gpus = torch.cuda.device_count()
print("GPU count:", n_gpus)
for i in range(n_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}, "
          f"{round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1)} GB VRAM")

if n_gpus < 1:
    raise RuntimeError("No GPU visible. Check NVIDIA drivers / CUDA install before continuing.")

CUDA available: True
GPU count: 2
  GPU 0: NVIDIA GeForce RTX 3090, 25.8 GB VRAM
  GPU 1: NVIDIA GeForce RTX 3090, 25.8 GB VRAM


In [2]:
# Cell 2: install the two packages this repo needs, if not already present.
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "medmnist", "tensorboardX", "ACSConv"],
    check=True,
)

import medmnist
print("medmnist version:", medmnist.__version__)

medmnist version: 3.0.2


In [3]:
# Cell 3: get the authors' own code, unmodified, as a ZIP (no working `git`
# needed on this machine). Skips re-downloading if already present.
import os
import urllib.request
import zipfile

REPO_DIR = "experiments-main"

if not os.path.exists(REPO_DIR):
    print("Downloading experiments repo...")
    urllib.request.urlretrieve(
        "https://github.com/MedMNIST/experiments/archive/refs/heads/main.zip",
        "experiments.zip",
    )
    with zipfile.ZipFile("experiments.zip", "r") as z:
        z.extractall(".")
    print("Extracted to", REPO_DIR)
else:
    print(f"{REPO_DIR} already exists, skipping download.")

MEDMNIST2D_DIR = os.path.join(REPO_DIR, "MedMNIST2D")
MEDMNIST3D_DIR = os.path.join(REPO_DIR, "MedMNIST3D")
assert os.path.isdir(MEDMNIST2D_DIR) and os.path.isdir(MEDMNIST3D_DIR), \
    "Expected MedMNIST2D/ and MedMNIST3D/ folders inside the repo, check the extraction."
print("2D script:", os.path.join(MEDMNIST2D_DIR, "train_and_eval_pytorch.py"))
print("3D script:", os.path.join(MEDMNIST3D_DIR, "train_and_eval_pytorch.py"))

experiments-main already exists, skipping download.
2D script: experiments-main\MedMNIST2D\train_and_eval_pytorch.py
3D script: experiments-main\MedMNIST3D\train_and_eval_pytorch.py


In [4]:
# Cell 4: fix a bug in the released code so --gpu_ids works for GPU 1 too, not just GPU 0.
import re

BUGGY_PATTERN = re.compile(r"torch\.device\('cuda:\{\}'\.format\(gpu_ids\[0\]\)\)")
FIXED_SNIPPET = "torch.device('cuda:0') if gpu_ids else torch.device('cpu')"

for script_path in [
    os.path.join(MEDMNIST2D_DIR, "train_and_eval_pytorch.py"),
    os.path.join(MEDMNIST3D_DIR, "train_and_eval_pytorch.py"),
]:
    with open(script_path, "r", encoding="utf-8") as f:
        content = f.read()

    if FIXED_SNIPPET in content:
        print(f"{script_path}: already patched, skipping.")
        continue

    matches = BUGGY_PATTERN.findall(content)
    if len(matches) != 1:
        print(f"{script_path}: WARNING, expected exactly 1 occurrence of the buggy "
              f"line, found {len(matches)}. Not modified; check manually before "
              f"trying to use GPU 1 with this script.")
        continue

    new_content = BUGGY_PATTERN.sub("torch.device('cuda:0')", content, count=1)
    explanation = (
        "# --- PATCHED 2026-09-03 (GPU-selection bugfix, see README) ---\n"
        "# Original code set CUDA_VISIBLE_DEVICES to mask in a single GPU, then addressed\n"
        "# it by its pre-mask index (only correct for GPU 0). After masking to any single\n"
        "# GPU, CUDA always re-numbers it as index 0, so the device line below now always\n"
        "# requests cuda:0; --gpu_ids still correctly controls which physical GPU gets\n"
        "# masked in. Zero effect on model/data/training procedure, only on which\n"
        "# physical chip a run executes on.\n"
    )
    new_content = new_content.replace("def main(", explanation + "def main(", 1)

    with open(script_path, "w", encoding="utf-8") as f:
        f.write(new_content)
    print(f"{script_path}: patched (device-selection line fixed).")

experiments-main\MedMNIST2D\train_and_eval_pytorch.py: already patched, skipping.
experiments-main\MedMNIST3D\train_and_eval_pytorch.py: already patched, skipping.


In [5]:
# Cell 5: dataset configuration, paper target numbers, and the command builder.
# This is the one cell worth reading before hitting "Run All".

NUM_EPOCHS = 20   # shortened from the paper's 100 (see the intro above); the LR
                  # drops scale with it, at 50% and 75% of NUM_EPOCHS.
NUM_RUNS = 5      # 5 independent runs per dataset.
GPU_IDS = [0, 1]  # both RTX 3090s.

# Table 3 (ResNet-50, 224px) and Table 4 (ResNet-50+3D) from the MedMNIST v2 paper.
PAPER_RESULTS = {
    # 2D
    "pathmnist":      {"AUC": 0.989, "ACC": 0.892},
    "chestmnist":     {"AUC": 0.773, "ACC": 0.948},
    "dermamnist":     {"AUC": 0.912, "ACC": 0.731},
    "octmnist":       {"AUC": 0.958, "ACC": 0.776},
    "pneumoniamnist": {"AUC": 0.962, "ACC": 0.884},
    "retinamnist":    {"AUC": 0.716, "ACC": 0.511},
    "breastmnist":    {"AUC": 0.866, "ACC": 0.842},
    "bloodmnist":     {"AUC": 0.997, "ACC": 0.950},
    "tissuemnist":    {"AUC": 0.932, "ACC": 0.680},
    "organamnist":    {"AUC": 0.998, "ACC": 0.947},
    "organcmnist":    {"AUC": 0.993, "ACC": 0.911},
    "organsmnist":    {"AUC": 0.975, "ACC": 0.785},
    # 3D
    "organmnist3d":     {"AUC": 0.994, "ACC": 0.883},
    "nodulemnist3d":    {"AUC": 0.875, "ACC": 0.847},
    "fracturemnist3d":  {"AUC": 0.725, "ACC": 0.494},
    "adrenalmnist3d":   {"AUC": 0.828, "ACC": 0.745},
    "vesselmnist3d":    {"AUC": 0.907, "ACC": 0.918},
    "synapsemnist3d":   {"AUC": 0.851, "ACC": 0.795},
}

DATASETS_2D = [
    "chestmnist", "pathmnist", "dermamnist", "octmnist", "pneumoniamnist",
    "retinamnist", "breastmnist", "bloodmnist", "tissuemnist",
    "organamnist", "organcmnist", "organsmnist",
]
DATASETS_3D = [
    "organmnist3d", "nodulemnist3d", "fracturemnist3d",
    "adrenalmnist3d", "vesselmnist3d", "synapsemnist3d",
]
ALL_DATASETS = DATASETS_2D + DATASETS_3D

# Only these two 3D datasets use --shape_transform (paper's regularization for
# "shape modality" datasets). Everything else omits it.
SHAPE_TRANSFORM_DATASETS = {"adrenalmnist3d", "vesselmnist3d"}


def build_command(data_flag, run_name, physical_gpu_id):
    """Returns (working_directory, command_list) for one training run."""
    if data_flag in DATASETS_3D:
        cmd = [
            "python", "train_and_eval_pytorch.py",
            "--data_flag", data_flag,
            "--model_flag", "resnet50",
            "--conv", "Conv3d",          # for "ResNet-50+3D" specifically
            "--pretrained_3d", "i3d",
            "--as_rgb",                  # paper: "copy the single channel into 3 channels"
            "--num_epochs", str(NUM_EPOCHS),
            "--batch_size", "32",        # paper's setting for 3D
            "--download",
            "--gpu_ids", str(physical_gpu_id),
            "--run", run_name,
            # Unique tree per run so two jobs starting in the same wall-clock
            # second cannot share the authors' "%y%m%d_%H%M%S" folder. On
            # Windows that collision is FileExistsError WinError 183; using
            # this supported --output_root flag, not a training-recipe change.
            "--output_root", os.path.join("output", run_name),
        ]
        if data_flag in SHAPE_TRANSFORM_DATASETS:
            cmd.append("--shape_transform")
        return MEDMNIST3D_DIR, cmd
    else:
        cmd = [
            "python", "train_and_eval_pytorch.py",
            "--data_flag", data_flag,
            "--model_flag", "resnet50",
            "--size", "28",
            "--resize",                  # reproduces the paper's "224" column
            "--as_rgb",                  # required for the resize path's torchvision ResNet
            "--num_epochs", str(NUM_EPOCHS),
            "--batch_size", "128",       # paper's setting for 2D
            "--download",
            "--gpu_ids", str(physical_gpu_id),
            "--run", run_name,
        ]
        return MEDMNIST2D_DIR, cmd


def find_result_csvs(data_flag, run_name):
    """Return existing test-set result CSVs for one (dataset, run), or [].

    Two layouts, both with a single * (Windows-safe; ** glob is unreliable
    on Windows and previously made this skip-check return nothing, which
    started a full 2D retrain):
      output/<dataset>/<timestamp>/...@runN.csv     (original 2D layout)
      output/<run>/<dataset>/<timestamp>/...@runN.csv  (3D unique-per-run)
    """
    import glob
    work_dir = MEDMNIST3D_DIR if data_flag in DATASETS_3D else MEDMNIST2D_DIR
    output_root = os.path.join(work_dir, "output")
    filename = f"{data_flag}_test_*@{run_name}.csv"
    matches = glob.glob(os.path.join(output_root, data_flag, "*", filename))
    matches += glob.glob(os.path.join(output_root, run_name, data_flag, "*", filename))
    return matches


print(f"Configured: {len(ALL_DATASETS)} datasets x {NUM_RUNS} runs = "
      f"{len(ALL_DATASETS) * NUM_RUNS} total jobs, NUM_EPOCHS={NUM_EPOCHS}, GPUs={GPU_IDS}")

Configured: 18 datasets x 5 runs = 90 total jobs, NUM_EPOCHS=20, GPUs=[0, 1]


In [6]:
# Cell 6: CALIBRATION. Run this before the main loop below. A real 5-epoch
# timed test on ChestMNIST (one of the larger datasets, so a reasonable
# worst-case), so the 20-vs-100 epoch decision is based on this machine's
# actual measured speed, not an estimate. Takes minutes, not hours.
import subprocess
import time

CALIBRATION_EPOCHS = 5
calibration_cmd = [
    "python", "train_and_eval_pytorch.py",
    "--data_flag", "chestmnist",
    "--model_flag", "resnet50",
    "--size", "28", "--resize", "--as_rgb",
    "--num_epochs", str(CALIBRATION_EPOCHS),
    "--batch_size", "128",
    "--download",
    "--gpu_ids", "0",
    "--run", "calibration",
]

print(f"Running a {CALIBRATION_EPOCHS}-epoch calibration on ChestMNIST (GPU 0)...")
t0 = time.time()
subprocess.run(calibration_cmd, cwd=MEDMNIST2D_DIR, check=True)
elapsed = time.time() - t0
per_epoch_min = elapsed / 60 / CALIBRATION_EPOCHS

print(f"\nMeasured: {elapsed/60:.1f} min for {CALIBRATION_EPOCHS} epochs "
      f"-> {per_epoch_min:.2f} min/epoch on this GPU, on ChestMNIST specifically "
      f"(a larger dataset, so this is close to a worst-case per-epoch cost; "
      f"smaller datasets will be much faster, TissueMNIST likely somewhat slower).")
print(f"\nAt this measured rate, for ChestMNIST alone (5 runs, 2 GPUs = 3 sequential slots):")
print(f"   20 epochs: {per_epoch_min * 20 * 3 / 60:.1f} hours")
print(f"  100 epochs: {per_epoch_min * 100 * 3 / 60:.1f} hours")
print("\nScale roughly by relative dataset size for a full-scope estimate across "
      "all 18 datasets. Use this real number, not the rough estimate above, to "
      "decide NUM_EPOCHS in the config cell before running the main loop below.")

Running a 5-epoch calibration on ChestMNIST (GPU 0)...

Measured: 29.5 min for 5 epochs -> 5.89 min/epoch on this GPU, on ChestMNIST specifically (a larger dataset, so this is close to a worst-case per-epoch cost; smaller datasets will be much faster, TissueMNIST likely somewhat slower).

At this measured rate, for ChestMNIST alone (5 runs, 2 GPUs = 3 sequential slots):
   20 epochs: 5.9 hours
  100 epochs: 29.5 hours

Scale roughly by relative dataset size for a full-scope estimate across all 18 datasets. Use this real number, not the rough estimate above, to decide NUM_EPOCHS in the config cell before running the main loop below.


In [6]:
# Cell 7: THE MAIN LOOP. Runs every (dataset, run) job that isn't already done,
# 2 at a time (one per GPU). This is the cell that actually takes hours; once
# started, everything else is automatic. Safe to stop and re-run: it always
# checks for existing results first and skips them.
import glob
import os
import queue
import subprocess
import threading
import time
from concurrent.futures import ThreadPoolExecutor

def output_glob(data_flag, run_name):
    """Re-bound here so this cell is safe to re-run after an interrupt
    even if Cell 5 was not re-executed in this kernel session."""
    work_dir = MEDMNIST3D_DIR if data_flag in DATASETS_3D else MEDMNIST2D_DIR
    return os.path.join(work_dir, "output", "**", f"{data_flag}_test_*@{run_name}.csv")

gpu_pool = queue.Queue()
for g in GPU_IDS:
    gpu_pool.put(g)

jobs = [(data_flag, run_idx) for data_flag in ALL_DATASETS for run_idx in range(1, NUM_RUNS + 1)]
print(f"Total jobs (datasets x runs): {len(jobs)}")

_lock = threading.Lock()
_completed = 0
_skipped = 0
_failed = 0

def run_job(job):
    global _completed, _skipped, _failed
    data_flag, run_idx = job
    run_name = f"run{run_idx}"

    existing = glob.glob(output_glob(data_flag, run_name), recursive=True)
    if existing:
        print(f"[{data_flag} {run_name}] already done, skipping.")
        with _lock:
            _skipped += 1
        return

    gpu_id = gpu_pool.get()
    try:
        work_dir, cmd = build_command(data_flag, run_name, gpu_id)
        if "--output_root" not in cmd:
            cmd.extend(["--output_root", os.path.join("output", run_name)])
        print(f"[{data_flag} {run_name}] starting on GPU {gpu_id}...")
        t0 = time.time()
        # capture_output so a failure's real error (e.g. a missing package's
        # traceback) is visible here instead of just an exit code, learned this
        # the hard way when all 3D jobs failed with no visible reason at first.
        result = subprocess.run(cmd, cwd=work_dir, capture_output=True, text=True)
        elapsed = time.time() - t0
        if result.returncode == 0:
            print(f"[{data_flag} {run_name}] done on GPU {gpu_id} in {elapsed/60:.1f} min.")
            with _lock:
                _completed += 1
        else:
            print(f"[{data_flag} {run_name}] FAILED on GPU {gpu_id} (exit "
                  f"{result.returncode}) after {elapsed/60:.1f} min. Last part of "
                  f"its error output:")
            print(result.stderr[-1500:] if result.stderr else "(no stderr captured)")
            print(f"[{data_flag} {run_name}] continuing with the remaining jobs regardless.")
            with _lock:
                _failed += 1
    finally:
        gpu_pool.put(gpu_id)

t_start = time.time()
with ThreadPoolExecutor(max_workers=len(GPU_IDS)) as pool:
    list(pool.map(run_job, jobs))
t_total = time.time() - t_start

print(f"\n=== This pass finished in {t_total/3600:.2f} hours ===")
print(f"Newly completed: {_completed}, already done: {_skipped}, failed: {_failed}, total jobs: {len(jobs)}")
if _failed > 0:
    print("Some jobs failed, scroll up for the specific dataset/run and error. "
          "Re-running this cell will retry only the ones that never produced output.")

Total jobs (datasets x runs): 90
[chestmnist run2] already done, skipping.
[chestmnist run1] already done, skipping.
[chestmnist run3] already done, skipping.
[chestmnist run4] already done, skipping.
[chestmnist run5] already done, skipping.
[pathmnist run1] already done, skipping.
[pathmnist run2] already done, skipping.
[pathmnist run3] already done, skipping.
[pathmnist run4] already done, skipping.
[pathmnist run5] already done, skipping.
[dermamnist run1] already done, skipping.
[dermamnist run2] already done, skipping.
[dermamnist run4] already done, skipping.
[dermamnist run3] already done, skipping.
[dermamnist run5] already done, skipping.
[octmnist run1] already done, skipping.
[octmnist run2] already done, skipping.
[octmnist run3] already done, skipping.
[octmnist run4] already done, skipping.
[octmnist run5] already done, skipping.
[pneumoniamnist run1] already done, skipping.
[pneumoniamnist run2] already done, skipping.
[pneumoniamnist run3] already done, skipping.
[pne

In [9]:
# Cell 8: aggregate everything found so far into one summary table + CSV.
# Safe to run any time, even mid-way through Cell 6, to check progress; re-run
# after Cell 6 finishes (or is stopped) for the real summary.
import csv
import glob
import re

import numpy as np

results_rows = []
all_complete = True

for data_flag in ALL_DATASETS:
    aucs, accs = [], []
    for run_idx in range(1, NUM_RUNS + 1):
        run_name = f"run{run_idx}"
        matches = find_result_csvs(data_flag, run_name)
        if matches:
            m = re.search(r"\[AUC\]([\d.]+)_\[ACC\]([\d.]+)", matches[0])
            if m:
                aucs.append(float(m.group(1)))
                accs.append(float(m.group(2)))

    paper = PAPER_RESULTS[data_flag]
    n_done = len(aucs)
    if n_done < NUM_RUNS:
        all_complete = False

    if n_done > 0:
        auc_mean, auc_std = float(np.mean(aucs)), float(np.std(aucs))
        acc_mean, acc_std = float(np.mean(accs)), float(np.std(accs))
        auc_gap = abs(auc_mean - paper["AUC"]) * 100
        acc_gap = abs(acc_mean - paper["ACC"]) * 100
        status = "COMPLETE" if n_done == NUM_RUNS else f"PARTIAL ({n_done}/{NUM_RUNS})"
    else:
        auc_mean = auc_std = acc_mean = acc_std = auc_gap = acc_gap = None
        status = "NOT STARTED"

    results_rows.append({
        "dataset": data_flag,
        "type": "3D" if data_flag in DATASETS_3D else "2D",
        "runs_done": n_done,
        "auc_mean": auc_mean, "auc_std": auc_std,
        "acc_mean": acc_mean, "acc_std": acc_std,
        "paper_auc": paper["AUC"], "paper_acc": paper["ACC"],
        "auc_gap_pts": auc_gap, "acc_gap_pts": acc_gap,
        "status": status,
    })

print(f"{'Dataset':<18}{'Type':<5}{'Runs':<7}{'AUC mean':<10}{'ACC mean':<10}{'GapAUC':<9}{'GapACC':<9}{'Status'}")
for r in results_rows:
    auc_m = f"{r['auc_mean']:.4f}" if r['auc_mean'] is not None else "-"
    acc_m = f"{r['acc_mean']:.4f}" if r['acc_mean'] is not None else "-"
    gap_a = f"{r['auc_gap_pts']:.2f}%" if r['auc_gap_pts'] is not None else "-"
    gap_c = f"{r['acc_gap_pts']:.2f}%" if r['acc_gap_pts'] is not None else "-"
    print(f"{r['dataset']:<18}{r['type']:<5}{str(r['runs_done'])+'/'+str(NUM_RUNS):<7}"
          f"{auc_m:<10}{acc_m:<10}{gap_a:<9}{gap_c:<9}{r['status']}")

with open("results_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(results_rows[0].keys()))
    writer.writeheader()
    writer.writerows(results_rows)
print("\nSaved results_summary.csv in the current folder, this is the file to send back.")

if all_complete:
    print("\n" + "=" * 50)
    print("ALL DATASETS COMPLETE")
    print("=" * 50)
else:
    remaining = sum(1 for r in results_rows if r["runs_done"] < NUM_RUNS)
    print(f"\n{remaining}/{len(ALL_DATASETS)} datasets still incomplete. "
          "Re-run Cell 7 above to continue (it skips whatever is already done), "
          "then this cell again to refresh the summary.")

Dataset           Type Runs   AUC mean  ACC mean  GapAUC   GapACC   Status
chestmnist        2D   5/5    0.7764    0.9474    0.34%    0.06%    COMPLETE
pathmnist         2D   5/5    0.9848    0.8888    0.42%    0.32%    COMPLETE
dermamnist        2D   5/5    0.9150    0.7408    0.30%    0.98%    COMPLETE
octmnist          2D   5/5    0.9676    0.7836    0.96%    0.76%    COMPLETE
pneumoniamnist    2D   5/5    0.9646    0.8556    0.26%    2.84%    COMPLETE
retinamnist       2D   5/5    0.7170    0.5128    0.10%    0.18%    COMPLETE
breastmnist       2D   5/5    0.8766    0.8410    1.06%    0.10%    COMPLETE
bloodmnist        2D   5/5    0.9980    0.9622    0.10%    1.22%    COMPLETE
tissuemnist       2D   5/5    0.9300    0.6830    0.20%    0.30%    COMPLETE
organamnist       2D   5/5    0.9970    0.9342    0.10%    1.28%    COMPLETE
organcmnist       2D   5/5    0.9934    0.9158    0.04%    0.48%    COMPLETE
organsmnist       2D   5/5    0.9780    0.7914    0.30%    0.64%    COMPLETE
o